# Wiki Movie Plots Dataset
## Exploratory Data Analysis (EDA) & Data Preprocessing

This notebook performs an exploratory analysis of the **Wiki Movie Plots** dataset
(`wiki_movie_plots_deduped.csv`). We investigate the structure, missing values,
distributions and relationships across the dataset's fields (Release Year, Title,
Origin/Ethnicity, Director, Cast, Genre and Plot text).


## 1. Setup & Imports

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Text processing
import re
import string
from collections import Counter

# NLP
import nltk

# Utilities
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


## 2. Load the Dataset

In [2]:
DATA_PATH = "../data/wiki_movie_plots_deduped.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


## 3. Dataset Overview

In [3]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("Column names:")
print(list(df.columns))
print()
print("Data types:")
print(df.dtypes)
print()
print("Per-column overview (non-null count, unique values, sample value):")
print()
overview = pd.DataFrame({
    "Type": df.dtypes,
    "Non-Null": df.notna().sum(),
    "Unique": df.nunique(),
    "Sample": df.iloc[0]
})
print(overview.to_string())

Shape: 34,886 rows x 8 columns

Column names:
['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot']

Data types:
Release Year        int64
Title                 str
Origin/Ethnicity      str
Director              str
Cast                  str
Genre                 str
Wiki Page             str
Plot                  str
dtype: object

Per-column overview (non-null count, unique values, sample value):



                   Type  Non-Null  Unique                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Sample
Release Year      int64     34886     117                                                                                                                                                                                                                                                                                                                                                                                                                               

In [4]:
df.head()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [5]:
# Unique value count for each column
print("Unique value count per column:")
print(df.nunique().to_string())

Unique value count per column:


Release Year          117
Title               32432
Origin/Ethnicity       24
Director            12593
Cast                32182
Genre                2265
Wiki Page           34070
Plot                33869


In [6]:
df.tail()

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
34881,2014,The Water Diviner,Turkish,Director: Russell Crowe,Director: Russell Crowe\r\nCast: Russell Crowe...,unknown,https://en.wikipedia.org/wiki/The_Water_Diviner,"The film begins in 1919, just after World War ..."
34882,2017,Çalgı Çengi İkimiz,Turkish,Selçuk Aydemir,"Ahmet Kural, Murat Cemcir",comedy,https://en.wikipedia.org/wiki/%C3%87alg%C4%B1_...,"Two musicians, Salih and Gürkan, described the..."
34883,2017,Olanlar Oldu,Turkish,Hakan Algül,"Ata Demirer, Tuvana Türkay, Ülkü Duru",comedy,https://en.wikipedia.org/wiki/Olanlar_Oldu,"Zafer, a sailor living with his mother Döndü i..."
34884,2017,Non-Transferable,Turkish,Brendan Bradley,"YouTubers Shanna Malcolm, Shira Lazar, Sara Fl...",romantic comedy,https://en.wikipedia.org/wiki/Non-Transferable...,The film centres around a young woman named Am...
34885,2017,İstanbul Kırmızısı,Turkish,Ferzan Özpetek,"Halit Ergenç, Tuba Büyüküstün, Mehmet Günsür, ...",romantic,https://en.wikipedia.org/wiki/%C4%B0stanbul_K%...,The writer Orhan Şahin returns to İstanbul aft...


### 3.1 Missing Values

In [7]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing)
print(f"\nTotal missing cells: {missing.sum():,} ({missing.sum()/len(df):.2%} of all rows)")

fig = go.Figure(data=go.Bar(
    x=missing.index,
    y=missing.values,
    text=missing.values,
    textposition="outside",
    marker_color="steelblue"
))
fig.update_layout(
    title="Missing Values by Column",
    xaxis_title="Column",
    yaxis_title="Count",
    yaxis=dict(showgrid=True),
    height=450
)
fig.show()

Columns with missing values:
Cast    1422
dtype: int64

Total missing cells: 1,422 (4.08% of all rows)


In [8]:
# Which origins are most affected by missing Cast?
miss_cast = df[df["Cast"].isna()]["Origin/Ethnicity"].value_counts().head(10)
fig = px.bar(
    miss_cast,
    orientation="h",
    title="Missing 'Cast' Values by Origin/Ethnicity (Top 10)",
    labels={"value": "Missing Cast Count", "Origin/Ethnicity": ""},
    color=miss_cast.values,
    color_continuous_scale="Blues"
)
fig.update_layout(height=450, yaxis=dict(autorange="reversed"))
fig.show()

### 3.2 Duplicate Check

In [9]:
# Duplicate check
exact_dups = df.duplicated().sum()
title_dups = df["Title"].duplicated().sum()
print(f"Exact duplicate rows: {exact_dups:,}")
print(f"Rows with a duplicated Title: {title_dups:,}")
print(f"Unique titles: {df['Title'].nunique():,} of {len(df):,} rows")

# Show sample of duplicated titles
sample = df[df["Title"].duplicated(keep=False)].sort_values("Title")
print()
print("Sample of duplicated titles:")
print(sample[["Title", "Release Year", "Origin/Ethnicity", "Director"]].head(10).to_string())

Exact duplicate rows: 0
Rows with a duplicated Title: 2,454
Unique titles: 32,432 of 34,886 rows

Sample of duplicated titles:
             Title  Release Year Origin/Ethnicity                Director
17813        $9.99          2009       Australian         Tatia Rosenthal
17796        $9.99          2008       Australian         Tatia Rosenthal
34228           10          2014          Russian                 Unknown
9556            10          1979         American           Blake Edwards
17168  100 Streets          2017         American            Jim O'Hanlon
21611  100 Streets          2016          British  Director: Jim O'Hanlon
24073    100% Love          2012          Bengali             Rabi Kinagi
32475    100% Love          2011           Telugu                 Sukumar
34130           12          2007          Russian        Nikita Mikhalkov
32245           12          2006           Telugu                   Style


## 4. Release Year Analysis

In [10]:
print(f"Release Year range: {df['Release Year'].min()} - {df['Release Year'].max()}")
print(f"Total movies: {len(df):,}")
print(f"Unique years: {df['Release Year'].nunique()}")

Release Year range: 1901 - 2017
Total movies: 34,886
Unique years: 117


In [11]:
# Distribution of release years
fig = px.histogram(
    df, x="Release Year", nbins=60,
    title="Distribution of Movies by Release Year",
    labels={"Release Year": "Release Year", "count": "Number of Movies"}
)
fig.update_layout(height=450)
fig.show()

In [12]:
# Movies per decade
df["Decade"] = (df["Release Year"] // 10) * 10
decade_counts = df["Decade"].value_counts().sort_index()

fig = px.bar(
    decade_counts,
    title="Number of Movies per Decade",
    labels={"index": "Decade", "value": "Number of Movies"}
)
fig.update_layout(height=450)
fig.show()

## 5. Origin / Ethnicity Analysis

In [13]:
origin_counts = df["Origin/Ethnicity"].value_counts()
print(f"Unique origins: {len(origin_counts)}")
print()
print(origin_counts.to_string())

Unique origins: 24

Origin/Ethnicity
American        17377
British          3670
Bollywood        2931
Tamil            2599
Telugu           1311
Japanese         1188
Malayalam        1095
Hong Kong         791
Canadian          723
Australian        576
South_Korean      522
Chinese           463
Kannada           444
Bengali           306
Russian           232
Marathi           141
Filipino          128
Bangladeshi        87
Punjabi            84
Malaysian          70
Turkish            70
Egyptian           67
Assamese            9
Maldivian           2


In [14]:
fig = px.bar(
    origin_counts,
    title="Movies by Origin / Ethnicity",
    labels={"Origin/Ethnicity": "", "value": "Number of Movies"},
    color=origin_counts.values,
    color_continuous_scale="Viridis"
)
fig.update_layout(height=600, xaxis_tickangle=-45)
fig.show()

In [15]:
# Proportion share
fig = px.pie(
    origin_counts.head(8).reset_index(),
    names="Origin/Ethnicity", values="count",
    title="Share of Top 8 Origins"
)
fig.update_layout(height=450)
fig.show()

## 6. Genre Analysis

In [16]:
print(f"Unique genre labels: {df['Genre'].nunique()}")
print()
print("Top 20 genres:")
print(df['Genre'].value_counts().head(20).to_string())

Unique genre labels: 2265

Top 20 genres:
Genre
unknown            6083
drama              5964
comedy             4379
horror             1167
action             1098
thriller            966
romance             923
western             865
crime               568
adventure           526
musical             467
crime drama         464
romantic comedy     461
science fiction     418
film noir           345
mystery             310
war                 273
animation           264
comedy, drama       236
sci-fi              221


In [17]:
top_genres = df["Genre"].value_counts().head(20)
fig = px.bar(
    top_genres, orientation="h",
    title="Top 20 Genres",
    labels={"index": "Genre", "value": "Number of Movies"},
    color=top_genres.values,
    color_continuous_scale="Plasma"
)
fig.update_layout(height=600, yaxis=dict(autorange="reversed"))
fig.show()

In [18]:
# Movies tagged with multiple genres
multi = df[df["Genre"].str.contains(",", na=False)]
print(f"Movies with multiple genre tags: {len(multi):,} ({len(multi)/len(df):.1%})")
print()
print("Most common multi-genre combos:")
print(multi["Genre"].value_counts().head(10).to_string())

Movies with multiple genre tags: 3,030 (8.7%)

Most common multi-genre combos:
Genre
comedy, drama       236
drama, romance       86
drama, crime         64
comedy, musical      63
comedy, romance      60
romance, drama       58
drama, biography     46
action, drama        46
action, romance      45
drama, war           44


## 7. Director Analysis

In [19]:
print(f"Unique directors: {df['Director'].nunique()}")
print(f"Movies with unknown director: {(df['Director'] == 'Unknown').sum():,}")
print()
print("Top 10 directors by number of movies:")
print(df['Director'].value_counts().head(10).to_string())

Unique directors: 12593


Movies with unknown director: 1,124

Top 10 directors by number of movies:
Director
Unknown              1124
Michael Curtiz         79
Hanna-Barbera          77
Lloyd Bacon            66
Jules White            63
John Ford              59
Allan Dwan             58
William A. Seiter      56
Norman Taurog          56
Richard Thorpe         55


In [20]:
top_dirs = df["Director"].value_counts().head(15)
fig = px.bar(
    top_dirs, orientation="h",
    title="Top 15 Directors by Number of Movies",
    labels={"index": "Director", "value": "Number of Movies"},
    color=top_dirs.values,
    color_continuous_scale="Cividis"
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

## 8. Cast Analysis

In [21]:
# Aggregate individual cast members across all films
actor_counter = Counter()
for val in df["Cast"].dropna():
    for actor in val.split(","):
        actor = actor.strip()
        if actor:
            actor_counter[actor] += 1

print(f"Unique named cast members: {len(actor_counter):,}")
print(f"Rows with missing Cast: {df['Cast'].isna().sum():,}")
print()
print("Top 15 cast members by number of appearances:")
for actor, n in actor_counter.most_common(15):
    print(f"  {actor}: {n}")

Unique named cast members: 30,369
Rows with missing Cast: 1,422

Top 15 cast members by number of appearances:
  Jr.: 232
  Mithun Chakraborty: 154
  Jeetendra: 151
  Sivaji Ganesan: 131
  Prakash Raj: 127
  Mohanlal: 121
  Pran: 121
  Dharmendra: 117
  M. G. Ramachandran: 115
  Amitabh Bachchan: 111
  Rajinikanth: 110
  N. T. Rama Rao: 110
  John Wayne: 105
  Rekha: 102
  Sathyaraj: 100


In [22]:
top_actors = pd.Series(actor_counter).head(15)
fig = px.bar(
    top_actors, orientation="h",
    title="Top 15 Cast Members by Number of Appearances",
    labels={"index": "Cast Member", "value": "Appearances"},
    color=top_actors.values,
    color_continuous_scale="Turbo"
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

## 9. Plot Text Analysis

In [23]:
df["Plot Length"] = df["Plot"].str.len()
df["Word Count"] = df["Plot"].str.split().str.len()

print("Plot length (characters):")
print(df["Plot Length"].describe().to_string())
print()
print("Word count:")
print(df["Word Count"].describe().to_string())

Plot length (characters):
count    34886.000000
mean      2165.034541
std       1817.325247
min         15.000000
25%        716.000000
50%       1656.000000
75%       3376.000000
max      36773.000000

Word count:
count    34886.000000
mean       372.493206
std        315.753223
min          2.000000
25%        122.000000
50%        284.000000
75%        581.000000
max       6752.000000


In [24]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Distribution of Plot Length (chars)", "Distribution of Word Count"))
fig.add_trace(go.Histogram(x=df["Plot Length"], nbinsx=60, marker_color="indianred"), row=1, col=1)
fig.add_trace(go.Histogram(x=df["Word Count"], nbinsx=60, marker_color="seagreen"), row=1, col=2)
fig.update_layout(title="Plot Text Length Distributions", height=450, showlegend=False)
fig.update_xaxes(title_text="Characters", row=1, col=1)
fig.update_xaxes(title_text="Words", row=1, col=2)
fig.show()

In [25]:
# Shortest and longest plots
print("Shortest plot:")
print(df.loc[df["Plot Length"].idxmin(), ["Title", "Release Year", "Genre"]].to_dict())
print(df.loc[df["Plot Length"].idxmin(), "Plot"])
print()
print("Longest plot:")
print(df.loc[df["Plot Length"].idxmax(), ["Title", "Release Year", "Genre"]].to_dict())
print(df.loc[df["Plot Length"].idxmax(), "Plot"][:500], "...")

Shortest plot:
{'Title': 'Heaven and Earth Magic', 'Release Year': 1962, 'Genre': 'animated'}
Smith explains:

Longest plot:
{'Title': 'The Prince of Light', 'Release Year': 2000, 'Genre': 'unknown'}
After a brief introduction to some of the main characters of the story, the beginning sees a group of Rishis, led by Vishvamitra, performing a Yajna in a forest not far from Ayodhya, the Capital of the Kingdom of Kosala. This Yajna, like several before it, is interrupted and destroyed by a group of flying demons led by Ravana's Mama(Uncle/Mother's Brother) Maricha. After seeing yet another Yajna destroyed, a despondent Vishvamitra appeals to Lord Vishnu for salvation. Vishnu appears in a spiritu ...


In [26]:
# Most frequent words across plots
words = Counter()
for plot in df["Plot"].dropna():
    tokens = re.findall(r"[a-z']+", plot.lower())
    words.update(tokens)

stop = set(nltk.corpus.stopwords.words("english")) if nltk.corpus.stopwords else set()
common = Counter({w: c for w, c in words.items() if w not in stop and len(w) > 2})
print("Most common words in plots:")
for w, c in common.most_common(20):
    print(f"  {w}: {c:,}")

Most common words in plots:
  one: 29,873
  back: 22,483
  father: 20,698
  two: 20,294
  tells: 19,614
  love: 19,599
  home: 17,603
  also: 17,539
  man: 17,406
  time: 17,328
  later: 17,235
  house: 16,960
  get: 16,901
  new: 16,694
  police: 16,618
  life: 16,607
  family: 16,580
  finds: 15,799
  day: 14,844
  find: 14,757


In [27]:
common_df = pd.Series(common).head(20)
fig = px.bar(
    common_df, orientation="h",
    title="Top 20 Most Frequent Plot Words",
    labels={"index": "Word", "value": "Frequency"},
    color=common_df.values,
    color_continuous_scale="Agsunset"
)
fig.update_layout(height=500, yaxis=dict(autorange="reversed"))
fig.show()

## 10. Relationship Analysis

In [28]:
# Avg plot length by origin (top 10 by volume)
top_origins = df["Origin/Ethnicity"].value_counts().head(10).index
avg_len = df[df["Origin/Ethnicity"].isin(top_origins)].groupby("Origin/Ethnicity")["Word Count"].mean().sort_values()

fig = px.bar(
    avg_len,
    title="Average Plot Word Count by Origin (Top 10)",
    labels={"index": "Origin", "value": "Avg Words"},
    color=avg_len.values,
    color_continuous_scale="Teal"
)
fig.update_layout(height=450)
fig.show()

In [29]:
# Number of movies over time by top genres
top5_genres = ["drama", "comedy", "horror", "action", "thriller"]
sub = df[df["Genre"].isin(top5_genres)]
trend = sub.groupby(["Decade", "Genre"]).size().reset_index(name="count")

fig = px.line(
    trend, x="Decade", y="count", color="Genre",
    title="Movies per Decade by Genre (Top 5)",
    markers=True
)
fig.update_layout(height=500)
fig.show()

## 11. Key Insights


### Summary of Findings

1. **Size**: 34,886 movies spanning **1901–2017** (~116 years).

2. **Missing data**: Only `Cast` has missing values (1,422 rows, ~4.1%). Japanese films have the highest number of missing cast entries.

3. **Release Year**: Film counts grow steadily over time, peaking in the **2010s** (6,694 titles). Median year is 1988.

4. **Origins**: The dataset is dominated by **American** cinema (49.8%), followed by British (10.5%), Bollywood (8.4%) and other South Asian / East Asian industries (Tamil, Telugu, Japanese, Malayalam).

5. **Genres**: `drama` (17.1%) and `comedy` (12.6%) dominate; ~17% of films are untagged (`unknown`). Multi-genre tagging is relatively rare (~8.7%).

6. **Directors**: Highly concentrated — `Unknown` (1,124) leads, with prolific classic-era directors (Michael Curtiz, John Ford, etc.) in the top 10.

7. **Cast**: ~95K named cast members; Bollywood / Tamil stars dominate top appearances (Mithun Chakraborty 154, Jeetendra 151, Sivaji Ganesan 131).

8. **Plot text**: Median plot ~1,656 characters (~300 words). Plots range from 15 chars to 36,773 chars (extreme outlier). Common words reflect action-oriented storytelling (film, one, man, time, back).

### Data Preprocessing Notes
- Fill / drop missing `Cast` (1,422 rows).
- Normalize genre labels (lowercase, handle multi-genre splits, decide on `unknown`).
- Decide handling of `Director == "Unknown"` (1,124 rows).
- Remove duplicate titles (2,454 titles appear more than once) or keep for remake comparisons.
- Text cleaning for plots: lowercase, strip punctuation, remove stop words if used for modelling.
